In [ ]:
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import scipy.stats as st
from matplotlib.transforms import ScaledTranslation

from chargers import *
from metrics import *
from graphs import *

## Abortion Experiment

In [ ]:
column_to_predict_abortion = "religion_14"
n_options_abortion = 3

d_abortion = {
  "abortion should be prohibited in all cases": 1,
  "always prohibited": 1,
  "always be prohibited": 1,
  "abortion should always be prohibited": 1,
  "abortion should always be prohibited.": 1,
  "abortion should be allowed in cases of special circumstances": 2,
  "abortion should be prohibited except in special cases": 2,
  "abortion should only be allowed in special cases.": 2,
  "abortion should only be allowed in special cases": 2,
  "abortion should be allowed only in special cases": 2,
  "abortion should be allowed in special cases": 2,
  "abortion should be only allowed in special cases": 2,
  "in special cases": 2,
  "abortion should be permitted in any case": 3,
  "abortion should be allowed in any case": 3,
  "abortion should be an option for women in any case.": 3,
  "abortion should be an option for women in any case": 3,
  "1": 1,
  "2": 2,
  "3": 3,
}

# Paths
abortion_route = "../results_dataframes/USA_abortion/"
experiments_route = "../../Data/Experiments/"
anes_file = abortion_route + "anes_2020_660.csv"

# Load religion_14 values from RF CSV (for models whose input file lacks it)
rf_abortion_full = pd.read_csv(experiments_route + "randomforest_3_eeuu.csv")
# RF abortion has 0-indexed religion_14 (0,1,2) -> shift to (1,2,3)
rf_abortion_full["religion_14"] = rf_abortion_full["religion_14"] + 1
religion_14_values = rf_abortion_full["religion_14"].values

In [ ]:
# --- Random Forest (abortion) ---
df_rf_abortion = rf_abortion_full.copy()
# pred values are already 1-indexed (1,2), keep as is

# --- Llama (abortion) --- pre-processed CSV already has religion_14 and pred
llama_abortion = pd.read_csv(experiments_route + "llama_aborto_eeuu.csv")

# --- GPT-3.5 (abortion) --- CSV already has 660 rows with religion_14 and raw pred text
gpt3_abortion_df = pd.read_csv(abortion_route + "abortion_usa_chatgpt3_cot.csv")
gpt3_abortion = data_charger(anes_file, gpt3_abortion_df, d_abortion, prompt_type='cot')
gpt3_abortion["religion_14"] = religion_14_values

# --- GPT-4 (abortion) ---
gpt4_abortion_df = pd.read_csv(abortion_route + "abortion_usa_chatgpt4_cot.csv")
gpt4_abortion = data_charger(anes_file, gpt4_abortion_df, d_abortion, prompt_type='cot')
gpt4_abortion["religion_14"] = religion_14_values

# --- Mistral (abortion) --- 659 lines JSONL
mistral_abortion = data_charger(anes_file, abortion_route + "abortion_usa_mistral_cot.jsonl",
                                d_abortion, prompt_type='cot', remove=659)
mistral_abortion["religion_14"] = religion_14_values[:659]

# --- T0 (abortion) --- 659 lines JSONL
t0_abortion = data_charger(anes_file, abortion_route + "abortion_usa_t0_cot.jsonl",
                           d_abortion, prompt_type='cot', remove=659, is_t0=True)
t0_abortion["religion_14"] = religion_14_values[:659]

print("Abortion data loaded:")
print(f"  RF: {len(df_rf_abortion)} rows")
print(f"  Llama: {len(llama_abortion)} rows")
print(f"  GPT-3.5: {len(gpt3_abortion)} rows")
print(f"  GPT-4: {len(gpt4_abortion)} rows")
print(f"  Mistral: {len(mistral_abortion)} rows")
print(f"  T0: {len(t0_abortion)} rows")

In [ ]:
# Compute metrics for abortion experiment
results_rf_abortion = metrics_dataset_gen_usa(df_rf_abortion, "religion_14", n_options=3, n_bootstrap=1000, random_seed=42)
results_llama_abortion = metrics_dataset_gen_usa(llama_abortion, "religion_14", n_options=3, n_bootstrap=1000, random_seed=42)
results_gpt3_abortion = metrics_dataset_gen_usa(gpt3_abortion, "religion_14", n_options=3, n_bootstrap=1000, random_seed=42)
results_gpt4_abortion = metrics_dataset_gen_usa(gpt4_abortion, "religion_14", n_options=3, n_bootstrap=1000, random_seed=42)
results_mistral_abortion = metrics_dataset_gen_usa(mistral_abortion, "religion_14", n_options=3, n_bootstrap=1000, random_seed=42)
results_t0_abortion = metrics_dataset_gen_usa(t0_abortion, "religion_14", n_options=3, n_bootstrap=1000, random_seed=42)

# Per-experiment plot: Llama vs RF
modelos_colores = {
    "Random Forest": "#5c6bc0",
    "ChatGPT-4": "#66bb6a",
    "ChatGPT-3.5": "#ff7043",
    "Llama-13b": "#ab47bc",
    "Mistral-Intruct-v0.2": "#ffa726",
    "T-zero": "#f48fb1",
}

sociodemographic_line_plot_usa(
    [results_llama_abortion, results_rf_abortion],
    modelos_colores,
    ["Llama-13b", "Random Forest"],
    "Abortion USA"
)

## Presidential Experiment

In [ ]:
column_to_predict_presidential = "elec_pres_144_a"
n_options_presidential = 4

d_presidential = {
    "joe biden": 1,
    "biden": 1,
    "donald trump": 2,
    "trump": 2,
    "jo jorgensen": 3,
    "jorgensen": 3,
    "howie hawkins": 4,
    "hawkins": 4,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
}

presidential_route = "../results_dataframes/USA_presidential/"

In [ ]:
# --- Random Forest (presidential) --- 0-indexed -> +1
df_rf_presidential = pd.read_csv(experiments_route + "randomforest_1_eeuu.csv")
df_rf_presidential["elec_pres_144_a"] = df_rf_presidential["elec_pres_144_a"] + 1
df_rf_presidential["pred"] = df_rf_presidential["pred"] + 1

# --- Llama (presidential) --- pre-processed CSV
llama_presidential = pd.read_csv(experiments_route + "llama_presidencial_eeuu.csv")

# --- GPT-3.5 (presidential) ---
gpt3_presidential_df = pd.read_csv(presidential_route + "presidential_usa_chatgpt3_cot.csv")
gpt3_presidential = data_charger(anes_file, gpt3_presidential_df, d_presidential, prompt_type='cot')

# --- GPT-4 (presidential) ---
gpt4_presidential_df = pd.read_csv(presidential_route + "presidential_usa_chatgpt4_cot.csv")
gpt4_presidential = data_charger(anes_file, gpt4_presidential_df, d_presidential, prompt_type='cot')

# --- Mistral (presidential) --- 660 lines JSONL
mistral_presidential = data_charger(anes_file, presidential_route + "presidential_usa_mistral_cot.jsonl",
                                    d_presidential, prompt_type='cot')

# --- T0 (presidential) --- 659 lines JSONL
t0_presidential = data_charger(anes_file, presidential_route + "presidential_usa_t0_cot.jsonl",
                               d_presidential, prompt_type='cot', remove=659, is_t0=True)

print("Presidential data loaded:")
print(f"  RF: {len(df_rf_presidential)} rows")
print(f"  Llama: {len(llama_presidential)} rows")
print(f"  GPT-3.5: {len(gpt3_presidential)} rows")
print(f"  GPT-4: {len(gpt4_presidential)} rows")
print(f"  Mistral: {len(mistral_presidential)} rows")
print(f"  T0: {len(t0_presidential)} rows")

In [ ]:
# Compute metrics for presidential experiment
results_rf_presidential = metrics_dataset_gen_usa(df_rf_presidential, "elec_pres_144_a", n_options=4, n_bootstrap=1000, random_seed=42)
results_llama_presidential = metrics_dataset_gen_usa(llama_presidential, "elec_pres_144_a", n_options=4, n_bootstrap=1000, random_seed=42)
results_gpt3_presidential = metrics_dataset_gen_usa(gpt3_presidential, "elec_pres_144_a", n_options=4, n_bootstrap=1000, random_seed=42)
results_gpt4_presidential = metrics_dataset_gen_usa(gpt4_presidential, "elec_pres_144_a", n_options=4, n_bootstrap=1000, random_seed=42)
results_mistral_presidential = metrics_dataset_gen_usa(mistral_presidential, "elec_pres_144_a", n_options=4, n_bootstrap=1000, random_seed=42)
results_t0_presidential = metrics_dataset_gen_usa(t0_presidential, "elec_pres_144_a", n_options=4, n_bootstrap=1000, random_seed=42)

# Per-experiment plot: Llama vs RF
sociodemographic_line_plot_usa(
    [results_llama_presidential, results_rf_presidential],
    modelos_colores,
    ["Llama-13b", "Random Forest"],
    "Presidential USA"
)

# Aggregated plots

In [ ]:
z = st.norm.ppf(0.975)  # ~1.96 for 95% CI
metric_cols = ["JSD", "Accuracy", "Harmonic Mean", "JSS", "Kappa"]

def aggregate_model_metrics(all_metrics_list):
    """Average metrics across experiments and combine CIs."""
    agg = all_metrics_list[0][["Group"]].copy()
    for col in metric_cols:
        ci_lo_col = f"{col}_CI_lower"
        ci_hi_col = f"{col}_CI_upper"
        means, ses = [], []
        for m in all_metrics_list:
            means.append(m[col].astype(float))
            ses.append((m[ci_hi_col].astype(float) - m[ci_lo_col].astype(float)) / (2 * z))
        mean_vals = sum(means) / len(means)
        se_combined = np.sqrt(sum(se**2 for se in ses)) / len(ses)
        agg[col] = mean_vals
        agg[ci_lo_col] = mean_vals - z * se_combined
        agg[ci_hi_col] = mean_vals + z * se_combined
    return agg

# Filter out Total row before aggregating
def no_total(df):
    return df[df['Group'] != 'Total'].reset_index(drop=True)

# Aggregate each model across the 2 experiments
agg_rf = aggregate_model_metrics([no_total(results_rf_abortion), no_total(results_rf_presidential)])
agg_llama = aggregate_model_metrics([no_total(results_llama_abortion), no_total(results_llama_presidential)])
agg_gpt3 = aggregate_model_metrics([no_total(results_gpt3_abortion), no_total(results_gpt3_presidential)])
agg_gpt4 = aggregate_model_metrics([no_total(results_gpt4_abortion), no_total(results_gpt4_presidential)])
agg_mistral = aggregate_model_metrics([no_total(results_mistral_abortion), no_total(results_mistral_presidential)])
agg_t0 = aggregate_model_metrics([no_total(results_t0_abortion), no_total(results_t0_presidential)])

print("=== Llama (USA aggregated) ===")
display(agg_llama)
print("\n=== Random Forest (USA aggregated) ===")
display(agg_rf)

In [ ]:
# Llama vs RF: Accuracy and JSS aggregated plot
fig, ax = plt.subplots(figsize=(12, 6))
plt.style.use('https://github.com/dhaitz/matplotlib-stylesheets/raw/master/pacoty.mplstyle')

grupos = {
    "Gender": (0, 1), "Age": (2, 4), "Zone": (5, 6),
    "Race": (7, 8), "Education": (9, 11),
    "GSE": (12, 14), "Ideology": (15, 18),
    "Party": (19, 21), "Religion": (22, 23),
}
colores_bg = {
    "Gender": "#f0f8ff", "Age": "#ffebcd", "Zone": "#add8e6",
    "Race": "#A49CF2", "Education": "#8FBC8F",
    "GSE": "#d3f8d3", "Ideology": "#ffe4e1",
    "Party": "#e0f7fa", "Religion": "#fffacd",
}
for grupo, (inicio, fin) in grupos.items():
    ax.axvspan(inicio - 0.5, fin + 0.5, color=colores_bg[grupo], alpha=0.5)

models = [
    {"data": agg_llama, "name": "Llama-13b"},
    {"data": agg_rf, "name": "Random Forest"},
]

metrics_to_plot = {
    "Accuracy": {"Llama-13b": "#e53935", "Random Forest": "#ef9a9a"},
    "JSS": {"Llama-13b": "#1e88e5", "Random Forest": "#90caf9"},
}

exclude_groups = ["No religion response", "Total"]

for metric, model_colors in metrics_to_plot.items():
    ci_lo = f"{metric}_CI_lower"
    ci_hi = f"{metric}_CI_upper"
    for model in models:
        data = model["data"].copy()
        data = data[~data['Group'].isin(exclude_groups)]
        data = data.reset_index(drop=True)

        color = model_colors[model["name"]]
        has_ci = ci_lo in data.columns and ci_hi in data.columns
        label = f"{metric} ({model['name']})"
        linestyle = '-' if model["name"] == "Llama-13b" else '--'

        for grupo, (inicio, fin) in grupos.items():
            x = list(range(inicio, fin + 1))
            y = data[metric].iloc[inicio:fin + 1].astype(float)
            ax.plot(x, y, linestyle + 'o', label=label if grupo == "Gender" else "",
                    color=color, markersize=4)
            if has_ci:
                y_lo = data[ci_lo].iloc[inicio:fin + 1].astype(float)
                y_hi = data[ci_hi].iloc[inicio:fin + 1].astype(float)
                ax.fill_between(x, y_lo, y_hi, color=color, alpha=0.12)

data_labels = agg_llama[~agg_llama['Group'].isin(exclude_groups)].reset_index(drop=True)

ax.set_ylim(0, 1)
ax.set_xticks(range(len(data_labels["Group"])))
ax.set_xticklabels(data_labels["Group"], rotation=80, ha='right')
ax.legend(loc='lower right', fontsize=9)
ax.set_title("Aggregated across experiments \u2014 Llama-13b vs Random Forest (USA)")

for label in ax.get_xticklabels():
    offset = ScaledTranslation(5/72, 0, fig.dpi_scale_trans)
    label.set_transform(label.get_transform() + offset)

plt.tight_layout()
plt.show()

In [ ]:
# Compute ratio Llama / RF with Delta Method CIs
ratio_df = agg_llama[["Group"]].copy()

for metric in ["Accuracy", "JSS", "Kappa"]:
    ci_lo = f"{metric}_CI_lower"
    ci_hi = f"{metric}_CI_upper"

    x = agg_llama[metric].astype(float)           # Llama
    y = agg_rf[metric].astype(float)               # RF
    se_x = (agg_llama[ci_hi].astype(float) - agg_llama[ci_lo].astype(float)) / (2 * z)
    se_y = (agg_rf[ci_hi].astype(float) - agg_rf[ci_lo].astype(float)) / (2 * z)

    r = x / y
    se_r = r * np.sqrt((se_x / x)**2 + (se_y / y)**2)

    ratio_df[metric] = r
    ratio_df[f"{metric}_CI_lower"] = r - z * se_r
    ratio_df[f"{metric}_CI_upper"] = r + z * se_r

ratio_df

In [ ]:
# Ratio plot (Llama / RF)
fig, ax = plt.subplots(figsize=(12, 7))
plt.style.use('https://github.com/dhaitz/matplotlib-stylesheets/raw/master/pacoty.mplstyle')

grupos = {
    "Gender": (0, 1), "Age": (2, 4), "Zone": (5, 6),
    "Race": (7, 8), "Education": (9, 11),
    "GSE": (12, 14), "Ideology": (15, 18),
    "Party": (19, 21), "Religion": (22, 23),
}
colores_bg = {
    "Gender": "#f0f8ff", "Age": "#ffebcd", "Zone": "#add8e6",
    "Race": "#A49CF2", "Education": "#8FBC8F",
    "GSE": "#d3f8d3", "Ideology": "#ffe4e1",
    "Party": "#e0f7fa", "Religion": "#fffacd",
}
for grupo, (inicio, fin) in grupos.items():
    ax.axvspan(inicio - 0.5, fin + 0.5, color=colores_bg[grupo], alpha=0.5)

# Filter rows
data = ratio_df[ratio_df['Group'] != "No religion response"]
data = data[data['Group'] != "Total"]
data = data.reset_index(drop=True)

# Plot each metric
metric_colors = {"Accuracy": "#1e35e5", "JSS": "#e53935"}
legend_names = {"Accuracy": "Relative Mean Accuracy", "JSS": "Relative Mean JSS"}
for metric, color in metric_colors.items():
    ci_lo = f"{metric}_CI_lower"
    ci_hi = f"{metric}_CI_upper"
    for grupo, (inicio, fin) in grupos.items():
        x = list(range(inicio, fin + 1))
        y = data[metric].iloc[inicio:fin + 1].astype(float)
        ax.plot(x, y, '-o', label=legend_names[metric] if grupo == "Gender" else "",
                color=color, markersize=4)
        y_lo = data[ci_lo].iloc[inicio:fin + 1].astype(float)
        y_hi = data[ci_hi].iloc[inicio:fin + 1].astype(float)
        ax.fill_between(x, y_lo, y_hi, color=color, alpha=0.15)

ax.set_ylim(bottom=0.5, top=1.4)
ax.set_xticks(range(len(data["Group"])))
ax.set_xticklabels(data["Group"], rotation=80, ha='right', fontsize=16)
ax.legend(loc='lower right', fontsize=14)
plt.yticks(fontsize=14)
ax.set_title("USA", fontsize=14)

plt.tight_layout()
fig.savefig("ratio_plot_usa.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# All-models Accuracy/JSS/Kappa plots
all_agg_models = [
    (agg_gpt3, "ChatGPT-3.5"),
    (agg_gpt4, "ChatGPT-4"),
    (agg_llama, "Llama-13b"),
    (agg_mistral, "Mistral-Intruct-v0.2"),
    (agg_rf, "Random Forest"),
]

grupos = {
    "Gender": (0, 1), "Age": (2, 4), "Zone": (5, 6),
    "Race": (7, 8), "Education": (9, 11),
    "GSE": (12, 14), "Ideology": (15, 18),
    "Party": (19, 21), "Religion": (22, 23),
}
colores_bg = {
    "Gender": "#f0f8ff", "Age": "#ffebcd", "Zone": "#add8e6",
    "Race": "#A49CF2", "Education": "#8FBC8F",
    "GSE": "#d3f8d3", "Ideology": "#ffe4e1",
    "Party": "#e0f7fa", "Religion": "#fffacd",
}
exclude_groups = ["No religion response", "Total"]

for metric in ["Accuracy", "JSS", "Kappa"]:
    fig, ax = plt.subplots(figsize=(14, 8))
    plt.style.use('https://github.com/dhaitz/matplotlib-stylesheets/raw/master/pacoty.mplstyle')

    for grupo, (inicio, fin) in grupos.items():
        ax.axvspan(inicio - 0.5, fin + 0.5, color=colores_bg[grupo], alpha=0.5)

    ci_lo_col = f"{metric}_CI_lower"
    ci_hi_col = f"{metric}_CI_upper"

    for model_data, model_name in all_agg_models:
        data = model_data.copy()
        data = data[~data['Group'].isin(exclude_groups)]
        data = data.reset_index(drop=True)
        has_ci = ci_lo_col in data.columns and ci_hi_col in data.columns
        color = modelos_colores[model_name]
        for grupo, (inicio, fin) in grupos.items():
            x = list(range(inicio, fin + 1))
            y = data[metric].iloc[inicio:fin + 1].astype(float)
            ax.plot(x, y, '-o', label=model_name if grupo == "Gender" else "",
                    color=color, markersize=4)
            if has_ci:
                y_lo = data[ci_lo_col].iloc[inicio:fin + 1].astype(float)
                y_hi = data[ci_hi_col].iloc[inicio:fin + 1].astype(float)
                ax.fill_between(x, y_lo, y_hi, color=color, alpha=0.12)

    ref_data = all_agg_models[0][0].copy()
    ref_data = ref_data[~ref_data['Group'].isin(exclude_groups)]
    ref_data = ref_data.reset_index(drop=True)

    ax.set_ylim(bottom=-0.02)
    ax.set_xticks(range(len(ref_data["Group"])))
    ax.set_xticklabels(ref_data["Group"], rotation=80, ha='right', fontsize=18)
    ax.legend(loc='lower right', fontsize=15)
    if metric == "Kappa":
        ax.legend(loc='center right', fontsize=15)
    ax.set_title(f"Aggregated {metric} across experiments \u2014 USA", fontsize=15)
    ax.set_ylabel(metric, fontsize=15)
    plt.yticks(fontsize=14)

    for label in ax.get_xticklabels():
        offset = ScaledTranslation(5/72, 0, fig.dpi_scale_trans)
        label.set_transform(label.get_transform() + offset)

    plt.tight_layout()
    fig.savefig(f"mean_{metric}_usa.pdf", bbox_inches='tight')
    plt.show()